In [0]:
from pyspark.sql import functions as F

In [0]:
#Catalog and table location
CATALOG = "global_electronics_retailer_dataset"


cust_df = spark.read.table(f"{CATALOG}.silver.customers")
exchange_df = spark.read.table(f"{CATALOG}.silver.exchange_rates")
prod_df = spark.read.table(f"{CATALOG}.silver.products")
sales_df = spark.read.table(f"{CATALOG}.silver.sales")
stores_df = spark.read.table(f"{CATALOG}.silver.stores")

In [0]:
#Remove ingestion time and source file names

cust_df = cust_df.drop("_sourcefile", "_ingest_time", "ingestion_date")
exchange_df = exchange_df.drop("_sourcefile", "_ingest_time","ingestion_date")
prod_df = prod_df.drop("_sourcefile", "_ingest_time","ingestion_date")
sales_df = sales_df.drop("_sourcefile", "_ingest_time", "ingestion_date")
stores_df = stores_df.drop("_sourcefile", "_ingest_time","ingestion_date")


In [0]:

prod_df = prod_df.withColumn("unit_price_usd", prod_df["unit_price_usd"]+50) 
display(prod_df)

In [0]:
# check if the rows are distinct in Product table
total_rows = prod_df.count()
print(f"Total number of rows in the products table: {total_rows}")
distinct_rows = prod_df.distinct().count()
print(f"Total number of distinct rows in the products table: {distinct_rows}")

if total_rows == distinct_rows:
  print("All rows are distinct")
else:
  print("There are duplicate rows")

In [0]:
# Create a smaller df with required fields for Join 
prod_lite_df = prod_df.select("product_key","unit_cost_usd","unit_price_usd")
fact_sales = sales_df.join(prod_lite_df,on="product_key",how="left")



In [0]:
fact_sales.printSchema()
print(sales_df.count())
print(fact_sales.count())

In [0]:
# Check for unmatched rows
print(fact_sales.filter(fact_sales["unit_cost_usd"].isNull()).count())
print(fact_sales.filter(fact_sales["unit_price_usd"].isNull()).count())
#Create a new column with the unit price in USD

In [0]:
fact_sales.explain()

In [0]:
fact_sales = fact_sales.withColumn("revenue_usd", fact_sales["quantity"] * fact_sales["unit_price_usd"])
fact_sales = fact_sales.withColumn("cost_usd", fact_sales["quantity"] * fact_sales["unit_cost_usd"])
fact_sales = fact_sales.withColumn("profit_usd", fact_sales["revenue_usd"] - fact_sales["cost_usd"])
fact_sales = fact_sales.withColumn("profit_margin", F.round(fact_sales["profit_usd"] / fact_sales["revenue_usd"],4))
fact_sales = fact_sales.withColumn("order_month", F.trunc(fact_sales["order_date"], "month"))

In [0]:
fact_sales= fact_sales.drop("unit_price_usd","unit_cost_usd")

In [0]:
display(fact_sales)

In [0]:
# Save the gold fact table
fact_sales.write.mode("overwrite")\
                    .option("overwriteSchema", "true")\
                    .saveAsTable(f"{CATALOG}.gold.fact_sales")

In [0]:
print(prod_df.count())
print(cust_df.count())
print(stores_df.count())
print(exchange_df.count())

In [0]:
print(prod_df.select("product_key").distinct().count())
print(cust_df.select("customer_key").distinct().count())
print(stores_df.select("store_key").distinct().count())
print(exchange_df.select("date").distinct().count())

In [0]:
display(exchange_df)

In [0]:
# Save dimension tables to the gold layer

prod_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_products")
cust_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_customers")
stores_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_stores")
exchange_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.dim_exchange_rates")

Build KPI tables for Dashboard

In [0]:

kpi_summary = fact_sales.agg(F.sum("revenue_usd").alias("total_revenue"),
                             F.sum("cost_usd").alias("total_cost"),
                             F.sum("profit_usd").alias("total_profit"),
                             F.countDistinct("order_number").alias("total_orders"))



In [0]:
kpi_summary = kpi_summary.withColumn("avg_order_value_usd",F.col("total_revenue")/F.col("total_orders"))
kpi_summary = kpi_summary.withColumn("Profit_Margin",F.round(F.col("total_profit")/F.col("total_revenue"),3))

In [0]:
display(kpi_summary)

In [0]:
kpi_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.kpi_summary")
